# **Homomorphic Encryption in Federated Learning (Paillier Cryptosystem)**

**Mathematical Foundations, Public-Key Generation, Additive Homomorphism, Fixed-Point Quantization, and Secure Model Aggregation in Pure Python & JAX**

---

### **1. Executive Summary & Theoretical Context**
In standard Federated Learning, an untrusted server directly inspects raw weight updates $\Delta \mathbf{w}_t^k$ uploaded by clients. While Differential Privacy adds protective noise (which permanently degrades model accuracy), **Homomorphic Encryption (HE)** allows a server to perform mathematical computations directly on **ciphertext without decrypting it**.

The **Paillier Cryptosystem** (Paillier, 1999) is an **additively homomorphic** public-key algorithm: multiplying two ciphertexts corresponds directly to adding their underlying plaintexts:

$$\mathcal{D}\left( \mathcal{E}(m_1) \cdot \mathcal{E}(m_2) \pmod{n^2} \right) = (m_1 + m_2) \pmod{n}$$

This property enables the central aggregator to compute the exact sum of client model updates $\sum_k \Delta \mathbf{w}_t^k$ while learning **zero information** about individual client parameters, providing an information-theoretic privacy shield with **zero loss in model accuracy**.


In [ ]:
from dotenv import find_dotenv, load_dotenv

# Search parent directories and load local environment variables
load_dotenv(find_dotenv("local.env"))

import math
import os
import random
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import seaborn as sns

from federated.privacy.paillier import (
    HomomorphicTensorCodec,
    PaillierPrivateKey,
    PaillierPublicKey,
    generate_paillier_keypair,
)
from models import count_parameters, get_model
from utils.data_processing import (
    configure_hf_cache_dir,
    load_processed_dataset,
)

# Enforce local disk caching for Hugging Face and Flower datasets
configure_hf_cache_dir()

# Plot styling configuration
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams.update({"font.size": 10, "figure.autolayout": True})

print("[✓] JAX backend initialized for Paillier Homomorphic Encryption simulation.")

## **2. Mathematical Formulation of the Paillier Cryptosystem**

### **1. Key Generation**
1. Choose two large independent prime numbers $p$ and $q$ of equal length.
2. Compute the RSA modulus $n = p \cdot q$ and Carmichael's function:
   $$\lambda = \text{lcm}(p - 1, q - 1) = \frac{(p - 1)(q - 1)}{\gcd(p - 1, q - 1)}$$
3. Set base $g = n + 1$, which simplifies the modular arithmetic since $(n + 1)^m \equiv 1 + m \cdot n \pmod{n^2}$.
4. Define function $L(u) = \frac{u - 1}{n}$. Compute the modular inverse:
   $$\mu = \left( L(g^\lambda \bmod n^2) \right)^{-1} \pmod{n}$$
- **Public Key**: $(n, g)$ (shared with all clients and the server).
- **Private Key**: $(\lambda, \mu)$ (kept strictly confidential by the Key Authority / Decryption Broker).

### **2. Encryption**
To encrypt integer message $m \in \mathbb{Z}_n$, choose random blinding factor $r \in \mathbb{Z}_n^*$:
$$c = \mathcal{E}(m) = g^m \cdot r^n \pmod{n^2}$$

### **3. Homomorphic Addition & Scalar Multiplication**
- **Sum of Ciphertexts**:
  $$c_{\text{sum}} = c_1 \cdot c_2 = (g^{m_1} r_1^n)(g^{m_2} r_2^n) = g^{m_1 + m_2} (r_1 r_2)^n \pmod{n^2} = \mathcal{E}(m_1 + m_2)$$
- **Scalar Multiplication by Plaintext $k$**:
  $$c^k = (g^m r^n)^k = g^{k \cdot m} (r^k)^n \pmod{n^2} = \mathcal{E}(k \cdot m)$$

### **4. Decryption**
$$m = \mathcal{D}(c) = L(c^\lambda \bmod n^2) \cdot \mu \pmod{n}$$

### **5. Fixed-Point Encoding for Real-Valued Tensors**
Neural network weights are real numbers $\theta \in \mathbb{R}$. We map them to integer plaintexts via fixed-point quantization with precision factor $S = 10^4$:
$$m = \text{int}(\theta \cdot S) \pmod{n}$$
Negative numbers $-\theta$ map cleanly via modular arithmetic: $(-\theta \cdot S) \equiv n - (\theta \cdot S) \pmod{n}$.


In [ ]:
# Generate Paillier keypair with primes p, q of specified bit-length
pub_test, priv_test = generate_paillier_keypair(prime_bits=64)
print(f"[✓] Paillier keypair generated: Modulus n = {pub_test.n.bit_length()} bits")

In [ ]:
# Verify homomorphic addition with signed negative numbers
v1 = 450
v2 = -180
v3 = 95

c1 = pub_test.encrypt(v1)
c2 = pub_test.encrypt(v2)
c3 = pub_test.encrypt(v3)

# Server adds ciphertexts blindly (without decrypting!)
c_sum = pub_test.add(pub_test.add(c1, c2), c3)

# Decrypt on client side
decrypted_sum = priv_test.decrypt(c_sum)
expected_sum = v1 + v2 + v3

print(f"[*] Encrypted inputs: {v1}, {v2}, {v3}")
print(f"[*] Decrypted aggregated sum: {decrypted_sum} (Expected: {expected_sum})")
assert decrypted_sum == expected_sum
print("[✓] Homomorphic addition & negative numbers mathematically verified!")

## **3. Encrypting Deep Model Updates with Fixed-Point Encoding**

We create a bridge between floating-point JAX PyTrees and encrypted Paillier ciphertexts.


In [ ]:
# Instantiate imported HomomorphicTensorCodec with public key
codec = HomomorphicTensorCodec(pub_test, precision_scale=1000)
print("[✓] Homomorphic Tensor Codec initialized.")

## **4. Federated Secure Aggregation Benchmark (Zero Information Leakage)**

We simulate a federated round with 5 edge clients updating a model layer.
The server computes the **exact aggregated global update** while seeing only ciphertext, guaranteeing zero gradient leakage with zero accuracy penalty.


In [ ]:
np.random.seed(42)
num_clients = 5
param_dim = 20

# Simulated gradient updates from 5 clients
client_updates = [
    np.random.normal(loc=0.0, scale=0.5, size=param_dim) for _ in range(num_clients)
]
true_sum = np.sum(client_updates, axis=0)

print(
    f"[*] Encrypting updates across {num_clients} edge clients ({param_dim} parameters each)..."
)
t0 = time.time()
encrypted_client_updates = [codec.encrypt_vector(u) for u in client_updates]
enc_time = time.time() - t0
print(f"[✓] Client-side encryption complete in {enc_time * 1000:.1f} ms")

# Server-side blind aggregation (server has NO private key!)
t0 = time.time()
server_aggregated_ciphers = codec.homomorphic_sum(encrypted_client_updates)
agg_time = time.time() - t0
print(f"[✓] Server-side blind encrypted sum complete in {agg_time * 1000:.1f} ms")

# Broker / Client decryption of the final global update
t0 = time.time()
decrypted_sum = codec.decrypt_vector(server_aggregated_ciphers, priv_test)
dec_time = time.time() - t0
print(f"[✓] Decryption of aggregate sum complete in {dec_time * 1000:.1f} ms")

# Numerical precision check
max_error = np.max(np.abs(decrypted_sum - true_sum))
mean_error = np.mean(np.abs(decrypted_sum - true_sum))
print("\n[★] Numerical Accuracy Verification:")
print(f"    Max Absolute Precision Error: {max_error:.6f}")
print(f"    Mean Absolute Error:         {mean_error:.6f}")
assert max_error < 0.01
print("[✓] Zero-leakage Federated Secure Aggregation successfully verified!")

## **5. Visualizing Exact Numerical Reconstruction & Cryptographic Security**

We plot the ground-truth plaintext updates against the homomorphically aggregated decrypted update across all parameter coordinates.


In [ ]:
indices = np.arange(param_dim)
plt.figure(figsize=(12, 5))

plt.plot(
    indices,
    true_sum,
    label="True Plaintext Sum",
    marker="o",
    linewidth=2,
    color="#1D3557",
)
plt.scatter(
    indices,
    decrypted_sum,
    label="Paillier Decrypted Sum",
    color="#E63946",
    s=70,
    zorder=5,
)

plt.title("Homomorphic Federated Aggregation: Exact Mathematical Reconstruction")
plt.xlabel("Model Parameter Coordinate Index")
plt.ylabel("Aggregated Gradient Value")
plt.legend()
plt.tight_layout()
plt.show()

# Inspect a raw ciphertext to illustrate cryptographic security to your supervisor
sample_cipher = encrypted_client_updates[0][0]
print("[*] What the central server actually sees (Raw Ciphertext):")
print(f"    {sample_cipher} (Length: {len(str(sample_cipher))} digits)")
print(
    "[✓] Cryptographically impossible to reverse-engineer client data from this cipher without lambda!"
)